# Hermiticity and complete positivity of the relaxation in MaRs

A triplet (S = 1, D = 500 MHz, E = 100 MHz) at 3.36 T, one crystal orientation, T = 5 K, microwave frequency 94 GHz.
The density matrix is propagated as in the propagator approach of MaRs: the propagator over one microwave period
(`mars.population.rk4.solve_matrix_ode_rk4`, 16 steps) is raised to the number of periods.

- **Hermiticity:** $\max|\rho_{ab}(t)-\rho^{*}_{ba}(t)|$ for the relaxation superoperator as built by MaRs.
- **Complete positivity:** the triplet is entangled with an identical copy, $\frac{1}{3}\sum_{ij}|i\rangle\langle j|\otimes|i\rangle\langle j|$, and only the triplet evolves. The joint density matrix has 9 eigenvalues with sum 1; at $t=0$ the largest is 1 and the others are 0. Complete positivity means that the smallest eigenvalue never becomes negative.

Two relaxation models: (A) Lindblad-type rates set in the xyz basis with the thermal correction; (B) Bloch–Redfield relaxation by librations about x, y and z with the spectral density corrected by Eq. (4.16). Each is shown as built by MaRs (non-secular) and secularized.

In [ ]:
import sys, os, math, warnings
# Delete this line if you installed MaRs.
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..', '..', '..')))
warnings.filterwarnings('ignore')
import numpy as np, torch
import matplotlib.pyplot as plt
import mars
from mars import spin_model, constants, population
from mars.population import tr_utils, rk4
from mars.population.transform import Liouvilleator

dtype = torch.float64; device = torch.device("cpu")
freq = 94e9                                  # microwave frequency (Hz), W-band
b1 = 1e-8                                    # microwave amplitude (T), the default of the propagator populator
n_steps = 16                                 # steps per microwave period

# ---------- a triplet, one crystal orientation ----------
g = spin_model.Interaction([2.000, 2.002, 2.004], dtype=dtype, device=device)
zfs = spin_model.DEInteraction([500e6, 100e6], dtype=dtype, device=device)
triplet = spin_model.SolidSample(
    base_spin_system=spin_model.SpinSystem(electrons=[1.0], g_tensors=[g], electron_electron=[(0, 0, zfs)]),
    ham_strain=0.0, gauss=0.002, lorentz=0.002, device=device, dtype=dtype)
ORI = 40
field = torch.tensor([3.36], dtype=dtype)
F, Gx, _, Gz = triplet.get_hamiltonian_terms()
energies, vectors = torch.linalg.eigh(F + Gz * field)            # [orientations, N], [orientations, N, N]
energies, vectors = energies[:, None], vectors[:, None]          # [orientations, 1, ...] as in MaRs
E, U = energies[ORI, 0], vectors[ORI, 0]
N = E.shape[-1]
Gx_eig = U.conj().T @ Gx[ORI].to(U.dtype) @ U                   # microwave operator in the eigenbasis (Hz/T)


def superop(context, temperature, kind):
    args = (vectors, None, field[None, :], energies, temperature)
    if kind == "thermal":
        return context.get_transformed_thermal_superop(*args)
    return context.get_transformed_driven_superop(*args)


def thermal_correction(R, temperature):
    return tr_utils.EvolutionSuper(energies).apply_thermal_correction(temperature, R)


def secular(R):
    """Keep only R_iijj and R_ijij."""
    keep = torch.eye(N * N, dtype=torch.bool)
    p = torch.arange(N) * (N + 1)
    keep[p[:, None], p[None, :]] = True
    return R * keep.to(R.dtype)


def rates(R):
    """w[i, j] = rate of the transfer j -> i = R_iijj."""
    p = torch.arange(N) * (N + 1)
    return R[p[:, None], p[None, :]].real


def period_propagator(R):
    """Propagator over one microwave period, as in the propagator approach of MaRs:
    L(t) = -i[H0 + 2π b1 Gx cos(ωt), .] + R, integrated by RK4 over 2π."""
    w = 2 * math.pi * freq
    static = Liouvilleator.hamiltonian_superop(torch.diag(2 * math.pi * E).to(torch.complex128)) + R.to(torch.complex128)
    dynamic = Liouvilleator.hamiltonian_superop(2 * math.pi * b1 * Gx_eig.to(torch.complex128))
    U2pi, _ = rk4.solve_matrix_ode_rk4(static / w, dynamic / w, n_steps)
    return U2pi


def propagate(U2pi, n_periods):
    """Map ρ(0) -> ρ(n τ) for every number of periods n."""
    return [torch.linalg.matrix_power(U2pi, int(n)) for n in n_periods]


def hermiticity(P_list, rho0):
    out = []
    for P in P_list:
        rho = (P @ rho0.reshape(-1)).reshape(N, N)
        out.append((rho - rho.conj().T).abs().max().item())
    return np.array(out)


def joint_eigenvalues(P_list):
    """The triplet is entangled with an identical copy, (1/N) Σ_ij |i><j| ⊗ |i><j|, and only the triplet
    evolves. Returns the N^2 eigenvalues of the joint density matrix for each time."""
    out = []
    for P in P_list:
        joint = P.reshape(N, N, N, N).permute(0, 2, 1, 3).reshape(N * N, N * N) / N
        out.append(torch.linalg.eigvalsh(0.5 * (joint + joint.conj().T)).numpy())
    return np.array(out)


def boltzmann(T):
    return torch.softmax(-constants.unit_converter(E, "Hz_to_K") / T, -1)


# initial state: |Tz> of the xyz basis in the eigenbasis
xyz = triplet.get_xyz_basis()[0].to(torch.complex128)
psi = U.to(torch.complex128).conj().T @ xyz[:, 2]
rho0 = torch.outer(psi, psi.conj())
tau = 1 / freq
n_periods = np.unique(np.logspace(1, np.log10(20e-6 / tau), 50).astype(np.int64))
t_us = n_periods * tau * 1e6


def panel_hermiticity(ax, U2pi):
    ax.semilogx(t_us, np.maximum(hermiticity(propagate(U2pi, n_periods), rho0), 1e-18), "-", color="C0")
    ax.axhline(2.2e-16, color="k", lw=0.8, ls=":", label="machine precision")
    ax.set(yscale="log", ylim=(1e-18, 1e-12), xlabel="t (µs)", ylabel=r"max $|\rho_{ab}-\rho^{*}_{ba}|$", title="Hermiticity")


In [ ]:

# ---------- Example A: Lindblad-type relaxation in the xyz basis + thermal correction ----------
T = torch.tensor(5.0, dtype=dtype)
ctx = population.Context(basis="xyz", sample=triplet, dtype=dtype, device=device,
                         thermal_rates=torch.tensor([[0, 1e6, 1e6], [1e6, 0, 1e6], [1e6, 1e6, 0]], dtype=dtype))
R_raw = superop(ctx, T, "thermal")
A_mars = thermal_correction(R_raw, T)[ORI, 0]                    # as computed in MaRs
A_sec = thermal_correction(secular(R_raw), T)[ORI, 0]            # secularized before the correction

# ---------- Example B: Bloch-Redfield relaxation, librations about x, y, z ----------
ops = []
for k in range(3):
    nvec = torch.zeros(3, dtype=dtype); nvec[k] = 1.0
    O_static, O_field = triplet.get_librations_along_axis(nvec)
    ops.append((O_static * 2 * math.pi, O_field * 2 * math.pi))


class SpectralDensity:
    """Lorentzian spectral density J(w) = A tau / (1 + (w tau)^2), corrected in MaRs by Eq. (4.16)."""
    def __init__(self, A, tau): self.A, self.tau = A, tau
    def __call__(self, w, temperature): return self.A * self.tau / (1 + (w * self.tau) ** 2)


def redfield(A):
    ch = [population.RedfieldRelaxationChannel(operator_components=[op], spectral_density_func=SpectralDensity(A, 1e-11),
                                               thermal_balance_mode="symmetric") for op in ops]
    c = population.Context(basis="product", sample=triplet, relaxation_coupling_channels=ch, dtype=dtype, device=device)
    return superop(c, T, "driven")[ORI, 0]


B_mars = redfield(1e6 / rates(redfield(1.0)).abs().max().item())    # largest rate 1e6 s^-1, non-secular
B_sec = secular(B_mars)

fig, axes = plt.subplots(2, 2, figsize=(11, 7.5))
for row, (name, R_mars, R_sec, lab_mars, lab_sec) in enumerate([
        ("Lindblad + thermal correction", A_mars, A_sec,
         "non-secular (as in MaRs)", "secularized before the correction"),
        ("Bloch–Redfield relaxation", B_mars, B_sec, "non-secular (as in MaRs)", "secular")]):
    U_mars, U_sec = period_propagator(R_mars), period_propagator(R_sec)
    panel_hermiticity(axes[row, 0], U_mars)
    axes[row, 0].set_title(f"Hermiticity: {name}", fontsize=10)
    for k, (U2pi, lab, sty) in enumerate([(U_mars, lab_mars, "-"), (U_sec, lab_sec, "--")]):
        ev = joint_eigenvalues(propagate(U2pi, n_periods))
        axes[row, 1].semilogx(t_us, ev[:, -1], sty, color=f"C{k}", lw=1, alpha=0.6, label=f"largest, {lab}")
        axes[row, 1].semilogx(t_us, ev[:, 0], sty, color=f"C{k}", lw=2, label=f"smallest, {lab}")
    axes[row, 1].axhline(0, color="k", lw=0.8)
    axes[row, 1].set(xlabel="t (µs)", ylabel="eigenvalue of the joint density matrix")
    axes[row, 1].set_title(f"Complete positivity: {name}", fontsize=10)
for a, l in zip(axes.flat, "abcd"):
    a.legend(fontsize=7); a.grid(alpha=0.3)
    a.text(-0.13, 1.03, f"({l})", transform=a.transAxes, fontweight="bold")
fig.tight_layout()
fig.savefig("properties.png", dpi=200)
plt.show()
